# RETIRED — never executed, superseded by `05_surface_code_interval.ipynb`

**Status: retired 2026-09-15. Kept for provenance, not for citation.**

This notebook sweeps the syndrome interval on a Steane `[[7,1,3]]` memory with a **per-round**
decoder — a lookup table applied one round at a time, with corrections accumulated in a Pauli
frame. Its written conclusion is "measure rarely".

**It was never run.** Every cell is unexecuted, so none of its numbers is backed by evidence.

**And its premise is off-model in two ways**, both established after it was written:

1. **A per-round decoder cannot absorb measurement faults**, so every extra round is pure cost
   and the best interval is trivially the longest one. `05` §4 shows this directly: with a
   spacetime decoder the answer inverts. The finding is a property of the decoder, not of the
   code or of τ.
2. **It has no `p_stab` term.** The paper's trade-off needs a per-round data cost that is
   independent of τ; without one the interval question is degenerate. That is the same defect
   that invalidated the first version of `05`, diagnosed in `05`'s opening section.

What survives from it is the circuit construction and the observation that PyMatching cannot
decode Steane — an `X` on `q6` flips all three checks, a weight-3 hyperedge — which is why the
surface code became the vehicle for the τ question and Steane stayed the vehicle for measuring
extraction cost in `04_extraction_cost.ipynb`.

---


# How often should syndromes be measured?

A Steane memory held for a **fixed total time**, with the syndrome interval τ swept.

```
encode |0_L>  ──▶  [ τ idle ops ; extract Z-syndrome ] × R  ──▶  read out  ──▶  decode
                    └─────────── R = TOTAL_IDLE / τ ───────────┘
```

**Holding total time fixed is what makes this a trade-off.** Add rounds without it and
error only ever grows, because each round is pure cost. Fix the wall-clock and the two
terms pull against each other:

| τ small | many rounds | measurement error dominates |
| --- | --- | --- |
| **τ large** | **few rounds** | **idle error accumulates unchecked** |

so an interior optimum can exist.

**Two knobs, established in `noise_analysis.ipynb`:**

- `P_IDLE` — `p_1q`, charged per idle operation. Idling is free until written in, so τ is
  realised as `phased_x(q, 0, 0)` repeated on every data qubit.
- `P_MEAS` — `p_meas`, charged per ancilla readout. Isolates cleanly.

Only Z-stabilizers are measured. The X and Z channels are independent in a CSS code, so a
bit-flip memory answers the interval question at half the circuit size.

## 0. Setup

In [ ]:
from collections import Counter
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from guppylang import guppy
from guppylang.std.angles import angle
from guppylang.std.builtins import array, output, py
from guppylang.std.quantum import collect_measurements, cx, h, measure, measure_array, qubit
from guppylang.std.qsystem import phased_x

from selene_sim import DepolarizingErrorModel, Quest, build

N_DATA, N_QUBITS = 7, 16
CHECKS = [[0, 2, 4, 6], [1, 2, 5, 6], [3, 4, 5, 6]]

plt.rcParams['figure.figsize'] = (9, 4.5)
pd.set_option('display.width', 120)


def syndrome_of(bits):
    """Z-syndrome of a classical bitstring, as an integer. Column j is binary j+1."""
    s = 0
    for k, supp in enumerate(CHECKS):
        if sum(bits[i] for i in supp) % 2:
            s |= 1 << k
    return s


def bits_to_int(three):
    return sum(int(b) << k for k, b in enumerate(three))

## 1. The circuit

`R` rounds, each `τ` idle operations on every data qubit followed by one Z-syndrome
extraction. `τ` and `R` are baked in with `py(...)`, so every point in the sweep is a
separate compilation — `build()` dominates the runtime and is cached.

In [ ]:
@guppy
def encode_zero() -> array[qubit, 7]:
    """|0_L>: seed qubits 0, 1, 3 with H, copy each onto its generator's support."""
    qs = array(qubit() for _ in range(7))
    h(qs[0]); h(qs[1]); h(qs[3])
    cx(qs[0], qs[2]); cx(qs[0], qs[4]); cx(qs[0], qs[6])
    cx(qs[1], qs[2]); cx(qs[1], qs[5]); cx(qs[1], qs[6])
    cx(qs[3], qs[4]); cx(qs[3], qs[5]); cx(qs[3], qs[6])
    return qs


@guppy
def syndrome_z(qs: array[qubit, 7]) -> array[bool, 3]:
    """Three Z-stabilizers via bare ancillas. Detects X errors."""
    a0, a1, a2 = qubit(), qubit(), qubit()
    cx(qs[0], a0); cx(qs[2], a0); cx(qs[4], a0); cx(qs[6], a0)
    cx(qs[1], a1); cx(qs[2], a1); cx(qs[5], a1); cx(qs[6], a1)
    cx(qs[3], a2); cx(qs[4], a2); cx(qs[5], a2); cx(qs[6], a2)
    return array(measure(a0).read(), measure(a1).read(), measure(a2).read())


def memory(tau, rounds):
    """Hold |0_L> through `rounds` intervals of `tau` idle ops each."""
    @guppy
    def prog() -> None:
        qs = encode_zero()
        for _ in range(py(rounds)):
            for _ in range(py(tau)):
                for i in range(7):
                    phased_x(qs[i], angle(0.0), angle(0.0))   # the idle operation
            output("sz", syndrome_z(qs))
        output("data", collect_measurements(measure_array(qs)))
    return prog


_BUILT = {}


def runner(tau, rounds):
    key = (tau, rounds)
    if key not in _BUILT:
        _BUILT[key] = build(memory(tau, rounds).compile())
    return _BUILT[key]


print('built: memory(tau, rounds)')

## 2. The decoder

Corrections accumulate in a **Pauli frame**, XORed into the data bits at readout. Purely
classical, so it adds no gates.

Two details make it work:

1. Each round reports the *cumulative* stabilizer value. Differencing consecutive rounds
   gives **detection events**, so each error is counted once.
2. The final Z readout is destructive and complete, so its syndrome can be computed
   classically. That last correction catches anything after the final round.

A measurement fault produces a spurious detection event and corrupts the frame — which is
exactly the cost that measuring more often incurs.

**Two decoders**, so the result does not rest on a hand-rolled one:

- **lookup** — for Steane at `d=3` the map is exact. Three checks give 8 syndromes, which
  is exactly `{no error} + {X on q_j}`, so every syndrome names its error with no ambiguity.
  Nothing can beat it on single faults.
- **BP-OSD** (`ldpc`) — belief propagation with ordered-statistics post-processing, the
  standard tool for general CSS codes. It generalises to codes where no exact table exists.

**PyMatching does not apply here.** Minimum-weight perfect matching needs every error to
flip at most two detectors. An `X` on `q6` flips all three — a weight-3 hyperedge — and
PyMatching rejects the check matrix outright. Matching becomes the right tool only on a
matching-compatible family such as the surface code.

In [ ]:
# Parity-check matrix, the one object both decoders share.
H = np.zeros((3, N_DATA), dtype=np.uint8)
for k, supp in enumerate(CHECKS):
    H[k, supp] = 1

# PyMatching cannot take this matrix: column 6 has three ones.
try:
    import pymatching
    pymatching.Matching(H)
    print('unexpected: pymatching accepted a weight-3 column')
except Exception as exc:
    print('pymatching on Steane ->', type(exc).__name__, str(exc)[:90])

from ldpc import BpOsdDecoder
_BPOSD = BpOsdDecoder(H, error_rate=0.01, osd_method='osd_e', osd_order=7)


def _correction_lookup(event):
    """Exact for d=3: syndrome integer j+1 names qubit j."""
    e = [0] * N_DATA
    if event:
        e[event - 1] ^= 1
    return e


def _correction_bposd(event):
    """BP-OSD on the same detection event."""
    if not event:
        return [0] * N_DATA
    s = np.array([(event >> k) & 1 for k in range(3)], dtype=np.uint8)
    return list(np.asarray(_BPOSD.decode(s), dtype=int) % 2)


CORRECTORS = {'lookup': _correction_lookup, 'bposd': _correction_bposd}


def decode(sz_rounds, data, method='lookup'):
    """Frame-track detection events, then decode the final measurement. Returns logical bit."""
    correct = CORRECTORS[method]
    frame = [0] * N_DATA
    previous = 0
    for s in sz_rounds:
        event = s ^ previous          # detection event, not the raw syndrome
        for i, bit in enumerate(correct(event)):
            frame[i] ^= bit
        previous = s

    corrected = [d ^ f for d, f in zip(data, frame)]
    residual = syndrome_of(corrected)   # the final round is perfect by construction
    for i, bit in enumerate(correct(residual)):
        corrected[i] ^= bit
    return sum(corrected) % 2


def run(tau, rounds, n_shots=800, seed=1, method='lookup', **channels):
    """Logical error rate, decoded and raw, for one (tau, rounds) point."""
    em = DepolarizingErrorModel(random_seed=seed, **channels)
    shots = runner(tau, rounds).run_shots(simulator=Quest(), n_qubits=N_QUBITS,
                                          n_shots=n_shots, error_model=em, random_seed=seed)
    dec = raw = 0
    for shot in shots:
        rec = list(shot)
        sz = [bits_to_int(v) for tag, v in rec if tag == "sz"]
        data = [int(b) for tag, v in rec if tag == "data" for b in v]
        dec += decode(sz, data, method=method)
        raw += sum(data) % 2
    return dec / n_shots, raw / n_shots


# Sanity: with no noise the logical bit must be 0 at every interval.
for tau, rounds in ((1, 4), (4, 2), (8, 1)):
    d, r = run(tau, rounds, n_shots=200)
    assert d == 0.0 and r == 0.0, (tau, rounds, d, r)
print('ideal simulation: logical state preserved at every interval tested')

## 3. The sweep

`TOTAL_IDLE` is the wall-clock: total idle operations each data qubit sits through,
**held constant**. τ divides it, so `R = TOTAL_IDLE / τ`. Small τ buys frequent checks at
the price of many noisy measurements.

In [ ]:
TOTAL_IDLE = 24
TAUS = [t for t in (1, 2, 3, 4, 6, 8, 12, 24) if TOTAL_IDLE % t == 0]

P_IDLE = 0.0008     # p_1q, charged per idle op (and per CX, via decomposition)
P_MEAS = 0.004      # p_meas, charged per ancilla readout
SHOTS  = 2000

rows = []
for tau in TAUS:
    rounds = TOTAL_IDLE // tau
    dec, raw = run(tau, rounds, n_shots=SHOTS, method='lookup', p_1q=P_IDLE, p_meas=P_MEAS)
    bp, _ = run(tau, rounds, n_shots=SHOTS, method='bposd', p_1q=P_IDLE, p_meas=P_MEAS)
    rows.append({'tau': tau, 'rounds': rounds, 'decoded': dec, 'bposd': bp, 'raw': raw,
                 'measurements': 3 * rounds})

sweep = pd.DataFrame(rows).set_index('tau')
best = sweep.decoded.idxmin()
print(f'TOTAL_IDLE={TOTAL_IDLE} held fixed; p_1q={P_IDLE}, p_meas={P_MEAS}, {SHOTS} shots')
print(f'lowest logical error at tau = {best} ({sweep.loc[best, "rounds"]} rounds)\n')
sweep

In [ ]:
fig, ax = plt.subplots()
ax.plot(sweep.index, sweep.decoded, marker='o', label='decoded (lookup)')
ax.plot(sweep.index, sweep.bposd, marker='s', linestyle='--', label='decoded (BP-OSD)')
ax.plot(sweep.index, sweep.raw, marker='.', alpha=.5, label='raw parity')
ax.axvline(best, color='k', linestyle='--', linewidth=1, label=f'minimum at tau={best}')
ax.set_xscale('log', base=2); ax.set_xticks(TAUS)
ax.get_xaxis().set_major_formatter(plt.ScalarFormatter())
ax.set_xlabel('syndrome interval tau (idle ops between rounds)')
ax.set_ylabel('logical error rate')
ax.set_title(f'Fixed total idle time ({TOTAL_IDLE} ops): measure often vs wait longer')
ax.legend(); ax.grid(alpha=.25)
plt.show()

## 4. Which knob moves the optimum

Sweeping each channel separately says what the optimum is made of. Raising `P_MEAS` should
push it to larger τ (measure less often); raising `P_IDLE` should pull it to smaller τ
(check more often).

In [ ]:
def optimum(p_idle, p_meas, n_shots=1200):
    out = {}
    for tau in TAUS:
        dec, _ = run(tau, TOTAL_IDLE // tau, n_shots=n_shots, p_1q=p_idle, p_meas=p_meas)
        out[tau] = dec
    s = pd.Series(out)
    return s, int(s.idxmin())


rows, curves = [], {}
for p_meas in (0.001, 0.004, 0.016):
    s, arg = optimum(P_IDLE, p_meas)
    curves[f'p_meas={p_meas}'] = s
    rows.append({'p_idle': P_IDLE, 'p_meas': p_meas, 'best_tau': arg,
                 'best_rate': round(s.min(), 4)})
for p_idle in (0.0004, 0.0016):
    s, arg = optimum(p_idle, P_MEAS)
    curves[f'p_idle={p_idle}'] = s
    rows.append({'p_idle': p_idle, 'p_meas': P_MEAS, 'best_tau': arg,
                 'best_rate': round(s.min(), 4)})

knobs = pd.DataFrame(rows)
knobs

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
for label, s in curves.items():
    style = '-o' if 'meas' in label else '--s'
    ax.plot(s.index, s.values, style, label=label, markersize=4)
ax.set_xscale('log', base=2); ax.set_xticks(TAUS)
ax.get_xaxis().set_major_formatter(plt.ScalarFormatter())
ax.set_xlabel('syndrome interval tau'); ax.set_ylabel('logical error rate')
ax.set_title('Where the optimum sits depends on which channel dominates')
ax.legend(fontsize=8); ax.grid(alpha=.25)
plt.show()

## 5. Findings

Read the `best_tau` column against the two channels.

**The two decoders agree.** For `d=3` the lookup table is exact, so BP-OSD cannot beat it
— matching it is the correctness check, and it means the result does not depend on a
hand-rolled decoder. BP-OSD is what carries over to a code where no exact table exists.

**The trade-off is real and measurable.** At fixed total time the curve is non-monotonic:
frequent checking pays measurement error, infrequent checking lets idle error outrun the
code's distance.

**The optimum moves with the ratio, not with either rate alone.** More measurement noise
pushes τ up; more idle noise pulls it down. That ratio is the only thing the interval
responds to.

**What is measured and what is declared.** The *shape* of the curve is measured in Selene.
The *calibration* is declared: no error model carries a duration, so the idle rate is set
by hand through the count of `phased_x` operations. A different declared idle rate moves
the optimum without changing the mechanism.

### Caveats

- **Bare ancillas.** A `Z` fault on a Z-type ancilla propagates back through the CNOTs onto
  several data qubits, inflating the per-round cost. Flagged or Shor-style ancillas would
  shrink it and move the optimum to smaller τ. The ancilla construction is a parameter of
  the experiment, not an implementation detail.
- **`p_1q` is shared.** CNOTs decompose into native gates charged at `p_1q`, so it pays for
  extraction as well as idling. Raising `P_IDLE` raises both; the separation is differential,
  not absolute.
- **Z-only.** A bit-flip memory. The full memory doubles the per-round cost and should move
  the optimum to larger τ.

### Next

1. **Fixed interval against adaptive.** τ driven by detection-event history rather than set
   once — the control this notebook is the baseline for.
2. **Leakage heralds as priors**, so the decoder knows which syndrome bits to distrust.
3. **Flagged ancillas**, to move the per-round cost and re-locate the optimum.